<a href="https://colab.research.google.com/github/arulselvi-git/arulselvi-codebooster-2026/blob/main/day5/day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
print('All libraries imported succssfully')

All libraries imported succssfully


In [16]:
df=pd.read_csv('student_performance.csv')
print(f'Dataset loaded:{df.shape[0]} students, {df.shape[1]} columns')
print(f'columns:{df.columns.tolist()}')
print(f'Missing values:{df.isnull().sum().sum()}')
print(f'\nFirst 5 rows:')
df.head()

Dataset loaded:30 students, 13 columns
columns:['student_id', 'name', 'age', 'gender', 'department', 'semester', 'math_score', 'science_score', 'english_score', 'programming_score', 'attendance_percentage', 'city', 'admission_year']
Missing values:0

First 5 rows:


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [17]:
print(df['programming_score'].describe())

count    30.000000
mean     67.600000
std      21.041175
min      38.000000
25%      49.250000
50%      66.000000
75%      88.750000
max      97.000000
Name: programming_score, dtype: float64


In [18]:
df_ml=df.copy()
le_gender=LabelEncoder()
df_ml['gender_encoded']=le_gender.fit_transform(df_ml['gender'])
print(f'Gender encoding:{dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_)))}')
le_dept=LabelEncoder()
df_ml['department_encoded']=le_dept.fit_transform(df_ml['department'])
print(f'Department encoding:{dict(zip(le_dept.classes_,le_dept.transform(le_dept.classes_)))}')
print('\nNew columns added:gender_encoded,department_encoded')
df_ml[['gender','gender_encoded','department','department_encoded']].head(5)

Gender encoding:{'Female': np.int64(0), 'Male': np.int64(1)}
Department encoding:{'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}

New columns added:gender_encoded,department_encoded


,gender,gender_encoded,department,department_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [19]:
feature_cols = [
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'department_encoded'
]

X = df_ml[feature_cols]
y = df_ml['programming_score']

print(f'Feature matrix X shape: {X.shape} (students x features)')
print(f'Target vector y shape: {y.shape} (one score per student)')

print(f'\nFeature columns: {feature_cols}')
print(f'Target column: programming_score')

print(f'\nTarget range: {y.min()} to {y.max()} (mean: {y.mean():.1f})')

Feature matrix X shape: (30, 6) (students x features)
Target vector y shape: (30,) (one score per student)

Feature columns: ['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'department_encoded']
Target column: programming_score

Target range: 38 to 97 (mean: 67.6)


In [20]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

print(f'Total students:{len(X)}')
print(f'Training students:{len(X_train)}({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing students:{len(X_test)}({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTraining target range:{y_train.min()}-{y_train.max()}')
print(f'Testing target range:{y_test.min()}-{y_test.max()}')

Total students:30
Training students:24(80%)
Testing students:6(20%)

Training target range:38-96
Testing target range:39-97


In [21]:
scaler = StandardScaler()

# Scale training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

print(f"Training feature mean (should be ≈ 0): {X_train_scaled.mean():.4f}")
print(f"Training feature std (should be ≈ 1): {X_train_scaled.std():.4f}")

Features scaled successfully!
Training feature mean (should be ≈ 0): -0.0000
Training feature std (should be ≈ 1): 1.0000


In [22]:


lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2= r2_score(y_test, lr_pred)

print('=== Model 1: Linear Regression ===')
print(f'MAE: {lr_mae:.2f} (prediction are off by {lr_mae:.1f} points on average)')
print(f'RMSE: {lr_rmse:.2f} (error with penalty for large mistakes)')
print(f'R^2: {lr_r2:.4f} ({lr_r2*100:.1f}% of programming score variation explained)')
print()
print('Learned coefficients (importance of each feature):')
for feat, coef in zip(feature_cols, lr_model.coef_):
  print(f'{feat}: {coef:+.3f}')
print(f'Bias (intercept): {lr_model.intercept_:.3f}')

=== Model 1: Linear Regression ===
MAE: 9.37 (prediction are off by 9.4 points on average)
RMSE: 11.47 (error with penalty for large mistakes)
R^2: 0.7356 (73.6% of programming score variation explained)

Learned coefficients (importance of each feature):
math_score: +20.256
science_score: +7.612
english_score: +2.396
attendance_percentage: -12.886
gender_encoded: -0.397
department_encoded: -0.449
Bias (intercept): 68.042


In [23]:
dt_model=DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_scaled,y_train)
dt_pred=dt_model.predict(X_test_scaled)

dt_mae=mean_absolute_error(y_test,dt_pred)
dt_rmse=np.sqrt(mean_squared_error(y_test,dt_pred))
dt_r2=r2_score(y_test,dt_pred)

print('=== Model 2: Decision Tree (max_depths)===')
print(f'MAE:{dt_mae:.2f}')
print(f'RMSE:{dt_rmse:.2f}')
print(f'R^2:{dt_r2:.4f}')

=== Model 2: Decision Tree (max_depths)===
MAE:10.75
RMSE:16.08
R^2:0.4803
